# Phase 2 — Zero-Shot kNN Baseline

**Approach:** Encode all training examples with frozen `all-MiniLM-L6-v2`. At inference, for each validation example compute mean cosine similarity to each label's training examples, then return the top-3 labels. Evaluated with MAP@3.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

from src.dataset import load_train, train_val_split, get_label_set
from src.evaluate import map_at_k, average_precision_at_k
from src.utils import encode_texts, cosine_similarity_matrix

DATA_DIR = Path('../data')
TRAIN_PATH = DATA_DIR / 'train_clean.csv'

df = load_train(TRAIN_PATH)
print(f'Total rows: {len(df)}')
print(f'Unique labels: {df["full_label"].nunique()}')
df.head(3)

In [ ]:
train_df, val_df = train_val_split(df, val_frac=0.2, seed=42)
label_set = get_label_set(train_df)

print(f'Train size: {len(train_df)}')
print(f'Val size:   {len(val_df)}')
print(f'Labels in train: {len(label_set)}')

## 2. Encode

In [ ]:
TRAIN_EMB_PATH  = DATA_DIR / 'train_embeddings.npy'
VAL_EMB_PATH    = DATA_DIR / 'val_embeddings.npy'
TRAIN_LABELS_PATH = DATA_DIR / 'train_labels.npy'
MODEL_NAME = 'all-MiniLM-L6-v2'

if TRAIN_EMB_PATH.exists() and VAL_EMB_PATH.exists() and TRAIN_LABELS_PATH.exists():
    print('Loading cached embeddings...')
    train_emb = np.load(TRAIN_EMB_PATH)
    val_emb   = np.load(VAL_EMB_PATH)
    train_labels_arr = np.load(TRAIN_LABELS_PATH, allow_pickle=True)
else:
    print('Encoding training set...')
    train_emb = encode_texts(train_df['model_input'].tolist(), model_name=MODEL_NAME)
    print('Encoding validation set...')
    val_emb   = encode_texts(val_df['model_input'].tolist(), model_name=MODEL_NAME)
    train_labels_arr = np.array(train_df['full_label'].tolist())

    np.save(TRAIN_EMB_PATH, train_emb)
    np.save(VAL_EMB_PATH, val_emb)
    np.save(TRAIN_LABELS_PATH, train_labels_arr)
    print('Saved embeddings to disk.')

print(f'train_emb shape: {train_emb.shape}')
print(f'val_emb shape:   {val_emb.shape}')

## 3. Retrieve — Per-Label Mean Similarity

In [ ]:
def predict_top_k(val_emb, train_emb, train_labels, label_set, k=3, chunk_size=512):
    """
    For each val example, compute mean cosine similarity to each label's
    training examples, then return top-k ranked labels.
    Processes val_emb in chunks to avoid OOM.
    """
    # Build label -> indices mapping
    label_to_idx = defaultdict(list)
    for i, lbl in enumerate(train_labels):
        label_to_idx[lbl].append(i)

    all_preds = []
    n_val = len(val_emb)

    for start in range(0, n_val, chunk_size):
        chunk = val_emb[start:start + chunk_size]  # (B, D)
        sim_matrix = cosine_similarity_matrix(chunk, train_emb)  # (B, N_train)

        for row_sims in sim_matrix:
            label_scores = {
                lbl: row_sims[idxs].mean()
                for lbl, idxs in label_to_idx.items()
            }
            top_k = sorted(label_scores, key=label_scores.get, reverse=True)[:k]
            all_preds.append(top_k)

        print(f'  Processed {min(start + chunk_size, n_val)}/{n_val} val examples', end='\r')

    print()
    return all_preds


print('Running kNN retrieval...')
val_preds = predict_top_k(val_emb, train_emb, train_labels_arr, label_set, k=3)
print(f'Example predictions (first 3):')
for i in range(3):
    print(f'  True: {val_df["full_label"].iloc[i]:<40}  Pred: {val_preds[i]}')

## 4. Evaluate — MAP@3

In [ ]:
y_true_list = val_df['full_label'].tolist()
overall_map, per_label = map_at_k(y_true_list, val_preds, k=3)

print(f'MAP@3 (overall): {overall_map:.4f}')
print(f'\nPer-label MAP@3 (top 10 best):')
sorted_labels = sorted(per_label.items(), key=lambda x: x[1], reverse=True)
for label, score in sorted_labels[:10]:
    print(f'  {label:<50} {score:.4f}')

## 5. Analysis

In [ ]:
# Per-label performance sorted by AP (worst labels)
print('Worst 10 labels by MAP@3:')
for label, score in sorted_labels[-10:]:
    count = y_true_list.count(label)
    print(f'  {label:<50} MAP={score:.4f}  (n={count})')

In [ ]:
# Confusion analysis: what does the model predict instead of the true label?
from collections import Counter

errors = [(true, pred[0]) for true, pred in zip(y_true_list, val_preds) if pred[0] != true]
wrong_pred_counts = Counter(pred for _, pred in errors)
print(f'Total errors (rank-1 wrong): {len(errors)} / {len(y_true_list)}')
print(f'\nMost common wrong top-1 predictions:')
for label, cnt in wrong_pred_counts.most_common(10):
    print(f'  {label:<50} {cnt}')

In [ ]:
# Sample failure cases
print('Sample failure cases (true label not in top-3):')
failures = [
    (i, y_true_list[i], val_preds[i])
    for i in range(len(y_true_list))
    if y_true_list[i] not in val_preds[i]
]
for i, true_lbl, preds in failures[:5]:
    text = val_df['model_input'].iloc[i][:120]
    print(f'\n[{i}] True: {true_lbl}')
    print(f'     Pred: {preds}')
    print(f'     Text: {text}...')

In [ ]:
# Check True_Correct:NA predictions
tc_label = 'True_Correct:NA'
tc_preds_as_top1 = sum(1 for p in val_preds if p[0] == tc_label)
tc_true_count = y_true_list.count(tc_label)
print(f'True_Correct:NA in val set: {tc_true_count} examples')
print(f'True_Correct:NA predicted as rank-1: {tc_preds_as_top1} times')
print(f'Per-label MAP@3 for True_Correct:NA: {per_label.get(tc_label, 0):.4f}')

## 6. Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Metric': 'MAP@3 (overall)', 'Value': f'{overall_map:.4f}'},
    {'Metric': 'Val set size', 'Value': str(len(val_df))},
    {'Metric': 'Train set size', 'Value': str(len(train_df))},
    {'Metric': 'Unique labels', 'Value': str(len(label_set))},
    {'Metric': 'Embedding model', 'Value': 'all-MiniLM-L6-v2 (frozen)'},
    {'Metric': 'Embedding dim', 'Value': '384'},
    {'Metric': 'Retrieval method', 'Value': 'kNN, per-label mean cosine sim'},
])
summary